## 1. Imports

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from scipy.io import loadmat

%matplotlib inline

## 2. Helper Functions (Padding, Convolution, Utilities)

In [ ]:
def pad_image(image, pad_h, pad_w, padding_type='zero'):
    """Vectorized padding for 2D grayscale images."""
    if padding_type == 'zero':
        return np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='constant')
    if padding_type == 'mirror':
        return np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='edge')
    raise ValueError("padding_type must be 'zero' or 'mirror'")

def convolve2d(image, kernel, padding_type='zero'):
    """Manual 2D convolution for odd-sized kernels (vectorized)."""
    kh, kw = kernel.shape
    if kh % 2 == 0 or kw % 2 == 0:
        raise NotImplementedError('Use dedicated function for even kernels (e.g., Roberts).')
    pad_h, pad_w = kh // 2, kw // 2
    padded = pad_image(image, pad_h, pad_w, padding_type)
    windows = np.lib.stride_tricks.sliding_window_view(padded, (kh, kw))
    return np.sum(windows * kernel, axis=(2, 3))

def convolve2x2(image, kernel, padding_type='zero'):
    """Manual convolution for 2x2 kernels (Roberts), vectorized."""
    kh, kw = kernel.shape
    if (kh, kw) != (2, 2):
        raise ValueError('Kernel must be 2x2.')
    mode = 'constant' if padding_type == 'zero' else 'edge'
    padded = np.pad(image, ((0, 1), (0, 1)), mode=mode)
    windows = np.lib.stride_tricks.sliding_window_view(padded, (2, 2))
    return np.sum(windows * kernel, axis=(2, 3))

def to_grayscale(image):
    if image.ndim == 3:
        image = np.dot(image[..., :3], [0.2989, 0.5870, 0.1140])
    if image.max() <= 1.0:
        image = (image * 255.0).astype(np.uint8)
    return image.astype(np.uint8)

## 3. Edge Operators (Sobel, Prewitt, Roberts, Laplacian)

In [ ]:
def sobel_gradients(image, padding_type='zero'):
    sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
    sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float32)
    gx = convolve2d(image, sobel_x, padding_type)
    gy = convolve2d(image, sobel_y, padding_type)
    mag = np.sqrt(gx ** 2 + gy ** 2)
    return mag, gx, gy

def prewitt_gradients(image, padding_type='zero'):
    prewitt_x = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], dtype=np.float32)
    prewitt_y = np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]], dtype=np.float32)
    gx = convolve2d(image, prewitt_x, padding_type)
    gy = convolve2d(image, prewitt_y, padding_type)
    mag = np.sqrt(gx ** 2 + gy ** 2)
    return mag

def roberts_gradients(image, padding_type='zero'):
    roberts_x = np.array([[1, 0], [0, -1]], dtype=np.float32)
    roberts_y = np.array([[0, 1], [-1, 0]], dtype=np.float32)
    gx = convolve2x2(image, roberts_x, padding_type)
    gy = convolve2x2(image, roberts_y, padding_type)
    mag = np.sqrt(gx ** 2 + gy ** 2)
    return mag

def laplacian_edge(image, padding_type='zero'):
    laplace = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float32)
    lap = convolve2d(image, laplace, padding_type)
    edge_map = np.abs(lap)
    return edge_map, lap

def laplacian_sharpening(image, c=1.0, padding_type='zero'):
    _, lap = laplacian_edge(image, padding_type)
    sharp = image - c * lap
    sharp = np.clip(sharp, 0, 255).astype(np.uint8)
    return sharp

## 4. Canny Edge Detection (Manual)

In [ ]:
def gaussian_kernel(size, sigma):
    ax = np.linspace(-(size // 2), size // 2, size)
    xx, yy = np.meshgrid(ax, ax)
    kernel = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    kernel = kernel / np.sum(kernel)
    return kernel

def canny_edge_detection(image, sigma=1.0, low_thresh_ratio=0.1, high_thresh_ratio=0.3, padding_type='zero'):
    
    # 1) Gaussian smoothing
    ksize = int(2 * np.ceil(3 * sigma) + 1)
    gauss = gaussian_kernel(ksize, sigma)
    smooth = convolve2d(image.astype(np.float32), gauss, padding_type)
    
    # 2) Gradients (Sobel)
    sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
    sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float32)
    gx = convolve2d(smooth, sobel_x, padding_type)
    gy = convolve2d(smooth, sobel_y, padding_type)
    mag = np.sqrt(gx ** 2 + gy ** 2)
    angle = np.arctan2(gy, gx) * 180.0 / np.pi
    angle[angle < 0] += 180

    # 3) Non-maximum suppression
    h, w = mag.shape
    suppressed = np.zeros_like(mag)
    for i in range(1, h - 1):
        for j in range(1, w - 1):
            theta = angle[i, j]
            if (0 <= theta < 22.5) or (157.5 <= theta <= 180):
                neighbors = (mag[i, j - 1], mag[i, j + 1])
            elif 22.5 <= theta < 67.5:
                neighbors = (mag[i - 1, j + 1], mag[i + 1, j - 1])
            elif 67.5 <= theta < 112.5:
                neighbors = (mag[i - 1, j], mag[i + 1, j])
            else:
                neighbors = (mag[i - 1, j - 1], mag[i + 1, j + 1])
            if mag[i, j] >= max(neighbors[0], neighbors[1]):
                suppressed[i, j] = mag[i, j]

    # 4) Hysteresis thresholding
    high = suppressed.max() * high_thresh_ratio
    low = high * low_thresh_ratio
    strong = 255
    weak = 100
    out = np.zeros_like(suppressed, dtype=np.uint8)
    strong_i, strong_j = np.where(suppressed >= high)
    weak_i, weak_j = np.where((suppressed < high) & (suppressed >= low))
    out[strong_i, strong_j] = strong
    out[weak_i, weak_j] = weak

    changed = True
    while changed:
        changed = False
        for i in range(1, h - 1):
            for j in range(1, w - 1):
                if out[i, j] == weak:
                    if (out[i + 1, j] == strong or out[i - 1, j] == strong or
                        out[i, j + 1] == strong or out[i, j - 1] == strong or
                        out[i + 1, j + 1] == strong or out[i - 1, j - 1] == strong or
                        out[i + 1, j - 1] == strong or out[i - 1, j + 1] == strong):
                        out[i, j] = strong
                        changed = True
    binary = (out == strong).astype(np.uint8) * 255
    return binary

## 5. Thresholding and Evaluation Metrics

In [ ]:
def otsu_threshold(image):
    hist, _ = np.histogram(image.ravel(), bins=256, range=(0, 256))
    total = image.size
    sum_total = np.sum(np.arange(256) * hist)
    sum_back, weight_back = 0, 0
    current_max, threshold = 0, 0
    for t in range(256):
        weight_back += hist[t]
        if weight_back == 0:
            continue
        weight_fore = total - weight_back
        if weight_fore == 0:
            break
        sum_back += t * hist[t]
        mean_back = sum_back / weight_back
        mean_fore = (sum_total - sum_back) / weight_fore
        between_var = weight_back * weight_fore * (mean_back - mean_fore) ** 2
        if between_var > current_max:
            current_max = between_var
            threshold = t
    return threshold

def binarize_edges(edge_response, threshold=None):
    edge_response = np.clip(edge_response, 0, 255).astype(np.uint8)
    if threshold is None:
        threshold = otsu_threshold(edge_response)
    return (edge_response >= threshold).astype(np.uint8) * 255

def precision_recall(detected, ground_truth):
    det = (detected > 0)
    gt = (ground_truth > 0)
    tp = np.sum(det & gt)
    fp = np.sum(det & ~gt)
    fn = np.sum(~det & gt)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return prec, rec

def localization_error(detected, ground_truth, max_dist=50):
    det_points = np.argwhere(detected > 0)
    gt_points = np.argwhere(ground_truth > 0)
    if len(det_points) == 0 or len(gt_points) == 0:
        return np.inf
    distances = []
    for d in det_points:
        diff = gt_points - d
        dists = np.sqrt(np.sum(diff ** 2, axis=1))
        min_d = np.min(dists)
        if min_d <= max_dist:
            distances.append(min_d)
    if not distances:
        return np.inf
    return float(np.mean(distances))

## 6. Load Images and Ground Truth

In [ ]:
img_dir = 'images/Task_1_Edge_Detection/'
jpg_files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith('.jpg')])
mat_files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith('.mat')])
print('JPG files:', jpg_files)
print('MAT files:', mat_files)

def extract_ground_truth(mat_data):
    # Try common keys first
    if 'groundTruth' in mat_data:
        gt = mat_data['groundTruth']
        # Handle structured arrays (BSDS style)
        if isinstance(gt, np.ndarray) and gt.dtype.names is not None:
            entry = gt[0, 0]
        elif isinstance(gt, np.ndarray) and gt.dtype == object:
            entry = gt[0, 0]
            if isinstance(entry, np.ndarray) and entry.dtype.names is not None:
                entry = entry[0, 0]
        else:
            entry = gt
        if isinstance(entry, np.void) and entry.dtype.names:
            if 'Boundaries' in entry.dtype.names:
                gt = entry['Boundaries']
            elif 'edge_map' in entry.dtype.names:
                gt = entry['edge_map']
            else:
                gt = entry[0]
        else:
            gt = entry
        return (np.array(gt) > 0).astype(np.uint8) * 255
    if 'edge_map' in mat_data:
        gt = mat_data['edge_map']
        return (gt > 0).astype(np.uint8) * 255
    # Fallback: first non-metadata key
    for k in mat_data.keys():
        if not k.startswith('__'):
            gt = mat_data[k]
            return (np.array(gt) > 0).astype(np.uint8) * 255
    raise ValueError('No valid ground truth found in .mat file')

data_items = []
for jpg_name in jpg_files:
    stem = os.path.splitext(jpg_name)[0]
    gt_name = next((m for m in mat_files if stem in m), None)
    if gt_name is None:
        raise FileNotFoundError('No matching ground truth .mat for ' + stem)
    img_path = os.path.join(img_dir, jpg_name)
    gt_path = os.path.join(img_dir, gt_name)
    image = to_grayscale(mpimg.imread(img_path))
    gt_data = loadmat(gt_path)
    gt = extract_ground_truth(gt_data)
    data_items.append({'name': stem, 'image': image, 'gt': gt})

print('Loaded images:', [d['name'] for d in data_items])
for item in data_items:
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1); plt.imshow(item['image'], cmap='gray'); plt.title(item['name'] + ' - Image')
    plt.subplot(1, 2, 2); plt.imshow(item['gt'], cmap='gray'); plt.title(item['name'] + ' - Ground Truth')
    plt.tight_layout()
    plt.show()

## 7. Apply Operators and Visual Comparison

In [ ]:
results = []
for item in data_items:
    name = item['name']
    image = item['image']
    gt = item['gt']

    sobel_mag, _, _ = sobel_gradients(image.astype(np.float32), padding_type='zero')
    prewitt_mag = prewitt_gradients(image.astype(np.float32), padding_type='zero')
    roberts_mag = roberts_gradients(image.astype(np.float32), padding_type='zero')
    laplace_edge, _ = laplacian_edge(image.astype(np.float32), padding_type='zero')
    sharpened = laplacian_sharpening(image.astype(np.float32), c=1.0, padding_type='zero')

    sobel_bin = binarize_edges(sobel_mag)
    prewitt_bin = binarize_edges(prewitt_mag)
    roberts_bin = binarize_edges(roberts_mag)
    laplace_bin = binarize_edges(laplace_edge)

    results.append({
        'name': name,
        'image': image,
        'gt': gt,
        'sobel_bin': sobel_bin,
        'prewitt_bin': prewitt_bin,
        'roberts_bin': roberts_bin,
        'laplace_bin': laplace_bin,
        'laplace_edge': laplace_edge,
        'sharpened': sharpened
    })

    plt.figure(figsize=(14, 8))
    plt.suptitle(name + ' - Edge Operators', y=1.02)
    plt.subplot(2, 3, 1); plt.imshow(sobel_mag, cmap='gray'); plt.title('Sobel')
    plt.subplot(2, 3, 2); plt.imshow(prewitt_mag, cmap='gray'); plt.title('Prewitt')
    plt.subplot(2, 3, 3); plt.imshow(roberts_mag, cmap='gray'); plt.title('Roberts')
    plt.subplot(2, 3, 4); plt.imshow(laplace_edge, cmap='gray'); plt.title('Laplacian Edge')
    plt.subplot(2, 3, 5); plt.imshow(sharpened, cmap='gray'); plt.title('Laplacian Sharpen')
    plt.subplot(2, 3, 6); plt.imshow(gt, cmap='gray'); plt.title('Ground Truth')
    plt.tight_layout()
    plt.show()

## 8. Quantitative Evaluation

In [ ]:
def evaluate_edges(name, detected, ground_truth):
    prec, rec = precision_recall(detected, ground_truth)
    f1 = 2 * prec * rec / (prec + rec + 1e-6)
    loc = localization_error(detected, ground_truth)
    print(f'{name}: P={prec:.3f}, R={rec:.3f}, F1={f1:.3f}, LocErr={loc:.2f}')

for res in results:
    print('\n' + res['name'] + ' - Metrics')
    evaluate_edges('Sobel', res['sobel_bin'], res['gt'])
    evaluate_edges('Prewitt', res['prewitt_bin'], res['gt'])
    evaluate_edges('Roberts', res['roberts_bin'], res['gt'])
    evaluate_edges('Laplacian', res['laplace_bin'], res['gt'])

## 9. Canny Multi-Scale Evaluation

In [ ]:
sigmas = [1.0, 2.0, 3.0]
for res in results:
    name = res['name']
    image = res['image']
    gt = res['gt']
    canny_maps = {}
    for s in sigmas:
        canny_maps[s] = canny_edge_detection(image, sigma=s, low_thresh_ratio=0.4, high_thresh_ratio=0.8)

    plt.figure(figsize=(12, 4))
    for i, s in enumerate(sigmas):
        plt.subplot(1, 3, i + 1)
        plt.imshow(canny_maps[s], cmap='gray')
        plt.title(f'{name} - Canny sigma={s}')
    plt.tight_layout()
    plt.show()

    for s in sigmas:
        evaluate_edges(f'Canny sigma={s}', canny_maps[s], gt)

## 10. Robustness to Gaussian Noise

In [ ]:
def add_gaussian_noise(image, sigma_noise=25):
    noise = np.random.normal(0, sigma_noise, image.shape)
    noisy = image.astype(np.float32) + noise
    return np.clip(noisy, 0, 255).astype(np.uint8)

for res in results:
    name = res['name']
    image = res['image']
    gt = res['gt']
    noisy_img = add_gaussian_noise(image, sigma_noise=30)
    plt.figure(figsize=(6, 4))
    plt.imshow(noisy_img, cmap='gray')
    plt.title(name + ' - Noisy Image (sigma=30)')
    plt.tight_layout()
    plt.show()

    sobel_mag_noisy, _, _ = sobel_gradients(noisy_img.astype(np.float32))
    sobel_bin_noisy = binarize_edges(sobel_mag_noisy)
    canny_noisy = canny_edge_detection(noisy_img, sigma=1.5, low_thresh_ratio=0.4, high_thresh_ratio=0.8)

    evaluate_edges('Sobel (noisy)', sobel_bin_noisy, gt)
    evaluate_edges('Canny (noisy)', canny_noisy, gt)